# OAI Package Inventory + Semiquant Venn
Thin notebook wrapper around `tmc_oai` production APIs.


In [7]:
from pathlib import Path
import math

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from matplotlib_venn import venn2
from IPython.display import clear_output, display

from tmc_oai import (
    build_package_inventory,
    build_semiquant_join,
    build_venn_payload,
    load_oai_env,
)


In [8]:
TIMEPOINT_MAP_CSV = None  # Optional override path; default comes from .oai_env.json
TOP_N_DEFAULT = 6


In [9]:
import pandas as pd

cfg = load_oai_env()
map_csv = Path(TIMEPOINT_MAP_CSV).expanduser().resolve() if TIMEPOINT_MAP_CSV else cfg.timepoint_map_csv

inventory_result = build_package_inventory(cfg.oai_dataset_root, map_csv)
semiquant_result = build_semiquant_join(cfg.oai_dataset_root, map_csv)

_timepoint_to_visit = {
    "BASELINE": "V00",
    "12MONTH": "V01",
    "18MONTH": "V02",
    "24MONTH": "V03",
    "30MONTH": "V05",
    "48MONTH": "V06",
    "72MONTH": "V08",
    "96MONTH": "V10",
}


def _visit_sort_key(value: str) -> tuple[int, int, str]:
    text = str(value).strip().upper()
    if text.startswith("V") and text[1:].isdigit():
        return (0, int(text[1:]), text)
    if text == "":
        return (2, 0, text)
    return (1, 0, text)


package_status = inventory_result.package_map.copy()
package_status["package_number"] = package_status["package_number"].astype(str)
package_status["timepoint_label"] = package_status["timepoint_label"].astype(str)
package_status["visit"] = package_status["timepoint_label"].map(_timepoint_to_visit).fillna("")
package_status["package_dir"] = package_status["package_number"].map(
    lambda value: cfg.oai_dataset_root / f"Package_{value}"
)
package_status["package_installed"] = package_status["package_dir"].map(lambda path: path.exists())
package_status["image03_available"] = package_status["package_dir"].map(
    lambda path: (path / "image03.txt").exists()
)

image_counts = inventory_result.category_counts.copy()
image_counts["package_number"] = image_counts["package_number"].astype(str)
image_counts["jpg"] = pd.to_numeric(image_counts["jpg"], errors="coerce").fillna(0)
image_counts["dicom"] = pd.to_numeric(image_counts["dicom"], errors="coerce").fillna(0)
image_counts = (
    image_counts.groupby("package_number", as_index=False)[["jpg", "dicom"]]
    .sum()
    .rename(columns={"jpg": "jpg_rows", "dicom": "dicom_rows"})
)
image_counts["has_any_images"] = (image_counts["jpg_rows"] + image_counts["dicom_rows"]) > 0

package_status = package_status.merge(
    image_counts[["package_number", "jpg_rows", "dicom_rows", "has_any_images"]],
    on="package_number",
    how="left",
)
package_status[["jpg_rows", "dicom_rows"]] = package_status[["jpg_rows", "dicom_rows"]].fillna(0).astype(int)
package_status["has_any_images"] = package_status["has_any_images"].astype("boolean").fillna(False).astype(bool)

package_status["disk_status"] = np.select(
    [
        ~package_status["package_installed"],
        package_status["package_installed"] & ~package_status["image03_available"],
        package_status["package_installed"] & package_status["image03_available"] & package_status["has_any_images"],
    ],
    [
        "package_missing",
        "image03_missing",
        "found_on_disk",
    ],
    default="metadata_only",
)

package_status = package_status.assign(_visit_order=package_status["visit"].map(_visit_sort_key))
package_status = package_status.sort_values(["_visit_order", "timepoint_label", "package_number"]).drop(columns=["_visit_order"])

print("Timepoint Label Mapping + On-Disk Status")
display(
    package_status[
        [
            "package_number",
            "timepoint_label",
            "visit",
            "disk_status",
            "package_installed",
            "image03_available",
            "jpg_rows",
            "dicom_rows",
        ]
    ]
)

display(inventory_result.coverage_summary)
display(semiquant_result.region_summary)
print(f"Collision assets across downloaded image packages: {len(semiquant_result.collision_assets):,}")
if not semiquant_result.collision_assets.empty:
    display(semiquant_result.collision_assets.head(25))




Timepoint Label Mapping + On-Disk Status


,package_number,timepoint_label,visit,disk_status,package_installed,image03_available,jpg_rows,dicom_rows
0,1243841,BASELINE,V00,found_on_disk,True,True,372,343
4,1243877,12MONTH,V01,found_on_disk,True,True,34,56
9,1244222,18MONTH,V02,metadata_only,True,True,0,0
8,1244220,24MONTH,V03,metadata_only,True,True,0,0
6,1244218,30MONTH,V05,metadata_only,True,True,0,0
5,1243880,48MONTH,V06,found_on_disk,True,True,4062,131
7,1244219,72MONTH,V08,metadata_only,True,True,0,0
3,1243872,96MONTH,V10,metadata_only,True,True,0,0
1,1243842,SCREENING,,package_missing,False,False,0,0
2,1243845,XRAYMETA,,image03_missing,True,False,0,0


,timepoint,meta,jpg,dicom,orphan_jpg,orphan_dicom,orphan_total
0,12MONTH,5901,34,56,0,0,0
1,18MONTH,0,0,0,0,0,0
2,24MONTH,5054,0,0,0,0,0
3,30MONTH,0,0,0,0,0,0
4,48MONTH,8123,4062,131,1694,571,1992
5,72MONTH,2157,0,0,0,0,0
6,96MONTH,4179,0,0,0,0,0
7,BASELINE,14982,372,343,0,0,0


,region,semiquant_rows,rows_with_jpeg,rows_with_dicom,rows_with_both,rows_missing_both,distinct_assets,collision_assets
0,Knee,57343,4235,541,128,52695,24537,0
1,Hip,16760,2754,292,76,13790,8380,0


Collision assets across downloaded image packages: 0


In [10]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display

regions = ["Hip", "Knee", "Hand"]

_TIMEPOINT_LABEL_TO_VISIT = {
    "BASELINE": "V00",
    "12MONTH": "V01",
    "18MONTH": "V02",
    "24MONTH": "V03",
    "30MONTH": "V05",
    "48MONTH": "V06",
    "72MONTH": "V08",
    "96MONTH": "V10",
}


def _label_to_visit(timepoint_label: str) -> str:
    label = str(timepoint_label).strip().upper()
    return _TIMEPOINT_LABEL_TO_VISIT.get(label, "")


def _safe_pct(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    num = pd.to_numeric(numerator, errors="coerce")
    den = pd.to_numeric(denominator, errors="coerce")
    out = pd.Series(np.nan, index=numerator.index, dtype="float64")
    mask = den > 0
    out.loc[mask] = (100.0 * num.loc[mask] / den.loc[mask]).astype(float)
    return out


def _sum_int(series: pd.Series) -> int:
    return int(pd.to_numeric(series, errors="coerce").fillna(0).sum())


def _sum_min_count(df: pd.DataFrame, group_cols: list[str], value_cols: list[str]) -> pd.DataFrame:
    grouped = (
        df.groupby(group_cols, as_index=False)[value_cols]
        .sum(min_count=1)
        .sort_values(group_cols)
        .reset_index(drop=True)
    )
    return grouped


def _append_total_row(
    df: pd.DataFrame,
    *,
    timepoint_label: str,
    region_label: str,
    status_label: str | None,
    sum_cols: list[str],
) -> pd.DataFrame:
    if df.empty:
        return df

    total_row = {col: pd.NA for col in df.columns}
    if "package_number" in total_row:
        total_row["package_number"] = "ALL_PACKAGES"
    if "timepoint_label" in total_row:
        total_row["timepoint_label"] = timepoint_label
    if "region" in total_row:
        total_row["region"] = region_label
    if "visit" in total_row:
        total_row["visit"] = "ALL"
    if status_label is not None and "status" in total_row:
        total_row["status"] = status_label

    for col in sum_cols:
        if col in df.columns:
            total_row[col] = _sum_int(df[col])

    if {"actual_any_images", "meta_rows", "any_pct_of_meta"}.issubset(df.columns):
        meta_total = _sum_int(df["meta_rows"])
        any_total = _sum_int(df["actual_any_images"])
        total_row["any_pct_of_meta"] = round((100.0 * any_total / meta_total), 1) if meta_total > 0 else np.nan

    return pd.concat([df, pd.DataFrame([total_row], columns=df.columns)], ignore_index=True)


package_map_df = inventory_result.package_map.copy()
package_map_df["package_number"] = package_map_df["package_number"].astype(str)
package_map_df["timepoint_label"] = package_map_df["timepoint_label"].astype(str)
package_map_df["package_dir"] = package_map_df["package_number"].map(
    lambda value: cfg.oai_dataset_root / f"Package_{value}"
)
package_map_df["package_installed"] = package_map_df["package_dir"].map(lambda path: path.exists())
package_map_df["image03_available"] = package_map_df["package_dir"].map(
    lambda path: (path / "image03.txt").exists()
)

coverage_raw = inventory_result.category_counts.copy()
coverage_raw["package_number"] = coverage_raw["package_number"].astype(str)
coverage_raw["timepoint_label"] = coverage_raw["timepoint"].astype(str)
coverage_raw["region"] = coverage_raw["category"].replace({"Pelvis": "Hip"})
coverage_raw = coverage_raw.loc[coverage_raw["region"].isin(regions)].copy()
coverage_raw["meta_rows"] = pd.to_numeric(coverage_raw["meta"], errors="coerce").fillna(0).astype(int)
coverage_raw["actual_jpeg_images"] = pd.to_numeric(coverage_raw["jpg"], errors="coerce").fillna(0).astype(int)
coverage_raw["actual_dicom_images"] = pd.to_numeric(coverage_raw["dicom"], errors="coerce").fillna(0).astype(int)

package_asset_counts = (
    coverage_raw.groupby("package_number", as_index=False)[["actual_jpeg_images", "actual_dicom_images"]]
    .sum()
)
package_asset_counts["package_has_any_disk_assets"] = (
    package_asset_counts["actual_jpeg_images"] + package_asset_counts["actual_dicom_images"]
) > 0
package_map_df = package_map_df.merge(
    package_asset_counts[["package_number", "package_has_any_disk_assets"]],
    on="package_number",
    how="left",
)
package_map_df["package_has_any_disk_assets"] = (
    package_map_df["package_has_any_disk_assets"].astype("boolean").fillna(False).astype(bool)
)

meta_by_key = (
    coverage_raw.groupby(["package_number", "timepoint_label", "region"], as_index=False)["meta_rows"]
    .sum()
)

venn_rows = inventory_result.venn_rows.copy()
if venn_rows.empty:
    observed_by_key = pd.DataFrame(
        columns=[
            "package_number",
            "timepoint_label",
            "region",
            "actual_jpeg_images",
            "actual_dicom_images",
            "actual_both_images",
            "actual_any_images",
            "actual_jpeg_only",
            "actual_dicom_only",
        ]
    )
else:
    venn_rows["timepoint_label"] = venn_rows["timepoint"].astype(str)
    venn_rows["package_number"] = (
        venn_rows["row_key"].astype(str).str.split(":", n=1).str[0]
    )
    venn_rows["region"] = venn_rows["category"].replace({"Pelvis": "Hip"})
    venn_rows = venn_rows.loc[venn_rows["region"].isin(regions)].copy()
    venn_rows["has_jpg_bool"] = venn_rows["has_jpg"].astype(bool)
    venn_rows["has_dicom_bool"] = venn_rows["has_dicom"].astype(bool)
    venn_rows["actual_jpeg_images"] = venn_rows["has_jpg_bool"].astype(int)
    venn_rows["actual_dicom_images"] = venn_rows["has_dicom_bool"].astype(int)
    venn_rows["actual_both_images"] = (venn_rows["has_jpg_bool"] & venn_rows["has_dicom_bool"]).astype(int)
    venn_rows["actual_any_images"] = (venn_rows["has_jpg_bool"] | venn_rows["has_dicom_bool"]).astype(int)
    venn_rows["actual_jpeg_only"] = (venn_rows["has_jpg_bool"] & ~venn_rows["has_dicom_bool"]).astype(int)
    venn_rows["actual_dicom_only"] = (venn_rows["has_dicom_bool"] & ~venn_rows["has_jpg_bool"]).astype(int)

    observed_by_key = (
        venn_rows.groupby(["package_number", "timepoint_label", "region"], as_index=False)[
            [
                "actual_jpeg_images",
                "actual_dicom_images",
                "actual_both_images",
                "actual_any_images",
                "actual_jpeg_only",
                "actual_dicom_only",
            ]
        ]
        .sum()
    )

report_grid = (
    package_map_df[[
        "package_number",
        "timepoint_label",
        "package_installed",
        "image03_available",
        "package_has_any_disk_assets",
    ]]
    .assign(key=1)
    .merge(pd.DataFrame({"region": regions, "key": 1}), on="key", how="inner")
    .drop(columns=["key"])
    .merge(meta_by_key, on=["package_number", "timepoint_label", "region"], how="left")
    .merge(observed_by_key, on=["package_number", "timepoint_label", "region"], how="left")
)

numeric_cols = [
    "meta_rows",
    "actual_jpeg_images",
    "actual_dicom_images",
    "actual_both_images",
    "actual_any_images",
    "actual_jpeg_only",
    "actual_dicom_only",
]
for col in numeric_cols:
    report_grid[col] = pd.to_numeric(report_grid[col], errors="coerce")

mask_image03 = report_grid["image03_available"].astype(bool)
report_grid.loc[mask_image03, numeric_cols] = report_grid.loc[mask_image03, numeric_cols].fillna(0)

report_grid["status"] = "package_missing"
mask_installed = report_grid["package_installed"].astype(bool)
mask_disk_assets = report_grid["package_has_any_disk_assets"].astype(bool)

report_grid.loc[mask_installed & ~mask_image03, "status"] = "image03_missing"
report_grid.loc[mask_installed & mask_image03 & mask_disk_assets, "status"] = "found_on_disk"
report_grid.loc[mask_installed & mask_image03 & ~mask_disk_assets, "status"] = "predicted_from_image03"
report_grid.loc[
    mask_installed & mask_image03 & report_grid["meta_rows"].isna(),
    "status",
] = "image03_parse_unavailable"

report_grid["visit"] = report_grid["timepoint_label"].map(_label_to_visit).replace("", pd.NA)

rate_source = report_grid.loc[
    report_grid["status"].eq("found_on_disk") & (report_grid["meta_rows"] > 0)
].copy()

if rate_source.empty:
    global_rates = {"jpg": 0.0, "dicom": 0.0, "both": 0.0}
else:
    total_meta = float(rate_source["meta_rows"].sum())
    global_rates = {
        "jpg": float(rate_source["actual_jpeg_images"].sum()) / total_meta,
        "dicom": float(rate_source["actual_dicom_images"].sum()) / total_meta,
        "both": float(rate_source["actual_both_images"].sum()) / total_meta,
    }

region_rates = {}
for region in regions:
    region_source = rate_source.loc[rate_source["region"].eq(region)]
    if region_source.empty or float(region_source["meta_rows"].sum()) <= 0:
        region_rates[region] = global_rates
        continue
    region_meta = float(region_source["meta_rows"].sum())
    region_rates[region] = {
        "jpg": float(region_source["actual_jpeg_images"].sum()) / region_meta,
        "dicom": float(region_source["actual_dicom_images"].sum()) / region_meta,
        "both": float(region_source["actual_both_images"].sum()) / region_meta,
    }

report_grid["predicted_jpeg_images"] = pd.NA
report_grid["predicted_dicom_images"] = pd.NA
report_grid["predicted_both_images"] = pd.NA
report_grid["predicted_any_images"] = pd.NA

mask_predict = report_grid["status"].eq("predicted_from_image03") & report_grid["meta_rows"].notna()
for region in regions:
    region_mask = mask_predict & report_grid["region"].eq(region)
    if not bool(region_mask.any()):
        continue

    meta = pd.to_numeric(report_grid.loc[region_mask, "meta_rows"], errors="coerce").fillna(0).astype(int)
    rates = region_rates.get(region, global_rates)

    pred_jpg = (meta * float(rates["jpg"])).round().astype(int)
    pred_dicom = (meta * float(rates["dicom"])).round().astype(int)
    pred_both = (meta * float(rates["both"])).round().astype(int)

    pred_both = np.minimum(pred_both, pred_jpg)
    pred_both = np.minimum(pred_both, pred_dicom)

    pred_any = pred_jpg + pred_dicom - pred_both
    pred_any = np.minimum(pred_any, meta)
    pred_any = np.maximum(pred_any, 0)

    report_grid.loc[region_mask, "predicted_jpeg_images"] = pred_jpg
    report_grid.loc[region_mask, "predicted_dicom_images"] = pred_dicom
    report_grid.loc[region_mask, "predicted_both_images"] = pred_both
    report_grid.loc[region_mask, "predicted_any_images"] = pred_any

report_grid["jpeg_images"] = pd.NA
report_grid["dicom_images"] = pd.NA
report_grid["both_images"] = pd.NA
report_grid["any_images"] = pd.NA

mask_found = report_grid["status"].eq("found_on_disk")
report_grid.loc[mask_found, "jpeg_images"] = report_grid.loc[mask_found, "actual_jpeg_images"]
report_grid.loc[mask_found, "dicom_images"] = report_grid.loc[mask_found, "actual_dicom_images"]
report_grid.loc[mask_found, "both_images"] = report_grid.loc[mask_found, "actual_both_images"]
report_grid.loc[mask_found, "any_images"] = report_grid.loc[mask_found, "actual_any_images"]

report_grid.loc[mask_predict, "jpeg_images"] = report_grid.loc[mask_predict, "predicted_jpeg_images"]
report_grid.loc[mask_predict, "dicom_images"] = report_grid.loc[mask_predict, "predicted_dicom_images"]
report_grid.loc[mask_predict, "both_images"] = report_grid.loc[mask_predict, "predicted_both_images"]
report_grid.loc[mask_predict, "any_images"] = report_grid.loc[mask_predict, "predicted_any_images"]

coverage_table_all = report_grid.loc[mask_found, [
    "package_number",
    "timepoint_label",
    "visit",
    "region",
    "meta_rows",
    "actual_jpeg_only",
    "actual_dicom_only",
    "actual_both_images",
    "actual_any_images",
]].copy()
coverage_table_all["any_pct_of_meta"] = _safe_pct(
    coverage_table_all["actual_any_images"],
    coverage_table_all["meta_rows"],
).round(1)
coverage_table_all = coverage_table_all.sort_values(["timepoint_label", "region"]).reset_index(drop=True)

prediction_table_all = report_grid[[
    "package_number",
    "timepoint_label",
    "visit",
    "region",
    "status",
    "meta_rows",
    "jpeg_images",
    "dicom_images",
    "both_images",
    "any_images",
]].copy()
prediction_table_all = prediction_table_all.sort_values(
    ["timepoint_label", "region"]
).reset_index(drop=True)


region_selector = widgets.Dropdown(
    options=["ALL"] + regions,
    value="ALL",
    description="Region",
)
report_out = widgets.Output()


def _filter_or_aggregate_coverage(df: pd.DataFrame) -> pd.DataFrame:
    if region_selector.value != "ALL":
        return df.loc[df["region"].eq(region_selector.value)].copy()

    value_cols = [
        "meta_rows",
        "actual_jpeg_only",
        "actual_dicom_only",
        "actual_both_images",
        "actual_any_images",
    ]
    all_view = _sum_min_count(
        df,
        group_cols=["package_number", "timepoint_label"],
        value_cols=value_cols,
    )
    all_view["region"] = "ALL"
    all_view["visit"] = all_view["timepoint_label"].map(_label_to_visit).replace("", pd.NA)
    all_view["any_pct_of_meta"] = _safe_pct(
        all_view["actual_any_images"],
        all_view["meta_rows"],
    ).round(1)
    cols = [
        "package_number",
        "timepoint_label",
        "visit",
        "region",
        "meta_rows",
        "actual_jpeg_only",
        "actual_dicom_only",
        "actual_both_images",
        "actual_any_images",
        "any_pct_of_meta",
    ]
    return all_view[cols]


def _filter_or_aggregate_prediction(df: pd.DataFrame) -> pd.DataFrame:
    if region_selector.value != "ALL":
        return df.loc[df["region"].eq(region_selector.value)].copy()

    value_cols = ["meta_rows", "jpeg_images", "dicom_images", "both_images", "any_images"]
    all_view = _sum_min_count(
        df,
        group_cols=["package_number", "timepoint_label", "status"],
        value_cols=value_cols,
    )
    all_view["region"] = "ALL"
    all_view["visit"] = all_view["timepoint_label"].map(_label_to_visit).replace("", pd.NA)
    cols = [
        "package_number",
        "timepoint_label",
        "visit",
        "region",
        "status",
        "meta_rows",
        "jpeg_images",
        "dicom_images",
        "both_images",
        "any_images",
    ]
    return all_view[cols]


def _render_reports(*_):
    with report_out:
        clear_output(wait=True)

        coverage_view = _filter_or_aggregate_coverage(coverage_table_all)
        prediction_view = _filter_or_aggregate_prediction(prediction_table_all)

        coverage_with_total = _append_total_row(
            coverage_view,
            timepoint_label="TOTAL_FOUND_ON_DISK",
            region_label=region_selector.value,
            status_label=None,
            sum_cols=["meta_rows", "actual_jpeg_only", "actual_dicom_only", "actual_both_images", "actual_any_images"],
        )

        prediction_with_total = _append_total_row(
            prediction_view,
            timepoint_label="TOTAL_AVAILABLE",
            region_label=region_selector.value,
            status_label="summary_total",
            sum_cols=["meta_rows", "jpeg_images", "dicom_images", "both_images", "any_images"],
        )

        print("Observed Coverage Table (found_on_disk only)")
        display(coverage_with_total)

        print("Prediction Table")
        display(prediction_with_total)


region_selector.observe(_render_reports, names="value")
display(region_selector, report_out)
_render_reports()




Dropdown(description='Region', options=('ALL', 'Hip', 'Knee', 'Hand'), value='ALL')

Output()

In [ ]:
import re
import pandas as pd

region_toggle = widgets.ToggleButtons(
    options=["Knee", "Hip"],
    value="Knee",
    description="Region",
)
timepoint_dropdown = widgets.Dropdown(
    options=[("ALL", "ALL")],
    value="ALL",
    description="Timepoint",
)
count_mode_toggle = widgets.ToggleButtons(
    options=[("Semiquant rows", "rows"), ("Unique asset_id", "assets")],
    value="rows",
    description="Count",
)
column_dropdown = widgets.Dropdown(options=[], description="Category")
venn_out = widgets.Output()

_NO_CATEGORY = "__NO_CATEGORY__"
_SEMIQ_FILE_BY_REGION = {
    "Knee": "oai_kxrsemiquant01.txt",
    "Hip": "oai_hxrsemiquant01.txt",
}


def _shorten_category_label(region: str, column_name: str, raw_description: str) -> str:
    text = " ".join(str(raw_description).split())
    if not text:
        return column_name

    text = re.sub(rf"^{re.escape(region)}\s+", "", text, flags=re.IGNORECASE)

    base_part = text.split("(", 1)[0].strip()
    grade_part = ""
    location_part = ""

    oarsi_match = re.search(r"OARSI\s*grades?\s*0\s*-\s*(\d+)", text, flags=re.IGNORECASE)
    generic_grade_match = re.search(r"grades?\s*0\s*-\s*(\d+)", text, flags=re.IGNORECASE)
    if oarsi_match:
        grade_part = f"OARSI 0-{oarsi_match.group(1)}"
    elif generic_grade_match:
        grade_part = f"0-{generic_grade_match.group(1)}"

    if ")" in text:
        location_part = text.split(")", 1)[1].strip()
    elif base_part != text:
        location_part = text.replace(base_part, "", 1).strip()

    location_part = location_part.replace("compartment", "").strip()
    location_part = re.sub(r"\s+", " ", location_part)

    location_replacements = {
        "lateral": "lat",
        "medial": "med",
        "supero-lateral": "supero-lat",
        "supero-medial": "supero-med",
        "joint space narrowing": "JSN",
        "subchondral": "subchr",
        "acetabular": "acetab",
        "femoral": "femoral",
        "tibia": "tibia",
        "femur": "femur",
    }
    for source, target in location_replacements.items():
        location_part = re.sub(rf"\b{re.escape(source)}\b", target, location_part, flags=re.IGNORECASE)

    label_parts = [base_part]
    if grade_part:
        label_parts.append(grade_part)
    if location_part:
        label_parts.append(location_part)

    short = " | ".join([part.strip() for part in label_parts if part.strip()])
    return short if short else column_name


def _load_description_lookup(region: str) -> dict[str, str]:
    file_name = _SEMIQ_FILE_BY_REGION.get(region, "")
    if not file_name:
        return {}

    file_path = semiquant_result.xraymeta_package_dir / file_name
    if not file_path.exists():
        return {}

    description_row = pd.read_csv(
        file_path,
        sep="	",
        dtype=str,
        keep_default_na=False,
        engine="python",
        nrows=1,
    )
    if description_row.empty:
        return {}

    return {
        str(column_name): str(description_row.iloc[0].get(column_name, "")).strip()
        for column_name in description_row.columns
    }


def _value_scale_key(raw_value: str) -> tuple[int, float, str]:
    value = str(raw_value).strip()
    lowered = value.lower()

    ordered_words = {
        "none": 0.0,
        "normal": 0.0,
        "mild": 1.0,
        "moderate": 2.0,
        "severe": 3.0,
        "very severe": 4.0,
    }
    for token, rank in ordered_words.items():
        if token in lowered:
            return (0, rank, lowered)

    numeric_match = re.search(r"-?\d+(?:\.\d+)?", value)
    if numeric_match:
        return (0, float(numeric_match.group(0)), lowered)

    if value == "<EMPTY>":
        return (2, 1e9, lowered)

    return (1, 1e9, lowered)


def _timepoint_sort_key(raw_value: str) -> tuple[int, int, str]:
    value = str(raw_value).strip()
    upper = value.upper()

    if upper in {"BASELINE", "BL", "V00", "0", "00"}:
        return (0, 0, upper)

    month_match = re.search(r"(\d+)\s*MONTH", upper)
    if month_match:
        return (1, int(month_match.group(1)), upper)

    digit_match = re.search(r"(\d+)", upper)
    if digit_match:
        return (2, int(digit_match.group(1)), upper)

    if upper == "<EMPTY>":
        return (4, 0, upper)

    return (3, 0, upper)


def _extract_timepoint_series(df: pd.DataFrame) -> pd.Series:
    candidates = ["visit", "visit_id", "timepoint", "timepoint_label"]
    for column_name in candidates:
        if column_name in df.columns:
            return df[column_name].astype(str).str.strip().replace("", "<EMPTY>")
    return pd.Series(["<EMPTY>"] * len(df), index=df.index)


_DEFAULT_VISIT_TO_TIMEPOINT_LABEL = {
    "V00": "BASELINE",
    "V01": "12MONTH",
    "V02": "18MONTH",
    "V03": "24MONTH",
    "V05": "30MONTH",
    "V06": "48MONTH",
    "V08": "72MONTH",
    "V10": "96MONTH",
}


def _build_visit_label_lookup() -> dict[str, str]:
    lookup: dict[str, str] = {}

    image_df = semiquant_result.image_xray_df.copy()
    if not image_df.empty and {"package_number", "visit"}.issubset(image_df.columns):
        visit_pkg = image_df[["package_number", "visit"]].copy()
        visit_pkg["package_number"] = visit_pkg["package_number"].astype(str).str.strip()
        visit_pkg["visit"] = visit_pkg["visit"].astype(str).str.strip()
        visit_pkg = visit_pkg.loc[visit_pkg["visit"].ne("")]

        package_map = inventory_result.package_map[["package_number", "timepoint_label"]].copy()
        package_map["package_number"] = package_map["package_number"].astype(str).str.strip()
        package_map["timepoint_label"] = package_map["timepoint_label"].astype(str).str.strip()

        merged = visit_pkg.merge(package_map, on="package_number", how="left")
        grouped = (
            merged.dropna(subset=["timepoint_label"])
            .groupby("visit")["timepoint_label"]
            .unique()
        )
        for visit_value, labels in grouped.items():
            clean_labels = sorted({str(label).strip() for label in labels if str(label).strip()})
            if len(clean_labels) == 1:
                lookup[str(visit_value)] = clean_labels[0]

    available_labels = set(
        inventory_result.package_map["timepoint_label"].astype(str).str.strip().tolist()
    )
    for visit_code, label in _DEFAULT_VISIT_TO_TIMEPOINT_LABEL.items():
        if visit_code not in lookup and label in available_labels:
            lookup[visit_code] = label

    return lookup


_VISIT_TO_TIMEPOINT_LABEL = _build_visit_label_lookup()


def _timepoint_option_label(visit_value: str) -> str:
    value = str(visit_value).strip()
    if value == "ALL":
        return "ALL"
    label = _VISIT_TO_TIMEPOINT_LABEL.get(value, "")
    if label:
        return f"{label} ({value})"
    return value


def _annotate_missing(ax, missing_count: int) -> None:
    ax.text(
        1.03,
        0.02,
        f"Missing both\n{int(missing_count):,}",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=9,
        clip_on=False,
        bbox={"facecolor": "white", "edgecolor": "#888888", "alpha": 0.85, "boxstyle": "round,pad=0.25"},
    )


def _draw_venn_or_message(
    ax,
    *,
    only_jpg: int,
    only_dicom: int,
    both: int,
    set_colors: tuple[str, str],
    alpha: float,
    empty_message: str,
) -> None:
    only_jpg = int(only_jpg)
    only_dicom = int(only_dicom)
    both = int(both)

    if (only_jpg + only_dicom + both) == 0:
        ax.axis("off")
        ax.text(
            0.5,
            0.58,
            "No JPEG/DICOM overlap",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=11,
            color="#555555",
        )
        ax.text(
            0.5,
            0.45,
            empty_message,
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=9,
            color="#777777",
        )
        return

    venn2(
        subsets=(only_jpg, only_dicom, both),
        set_labels=("JPEG", "DICOM"),
        set_colors=set_colors,
        alpha=alpha,
        ax=ax,
    )


def _asset_counts(frame: pd.DataFrame) -> tuple[dict[str, int], int]:
    if frame.empty:
        return {"only_jpg": 0, "only_dicom": 0, "both": 0, "missing_both": 0}, 0

    asset_presence = (
        frame.groupby("asset_id", as_index=False)
        .agg(has_jpg=("has_jpg", "max"), has_dicom=("has_dicom", "max"))
    )
    jpg = asset_presence["has_jpg"].fillna(False).astype(bool)
    dicom = asset_presence["has_dicom"].fillna(False).astype(bool)

    counts = {
        "only_jpg": int((jpg & ~dicom).sum()),
        "only_dicom": int((dicom & ~jpg).sum()),
        "both": int((jpg & dicom).sum()),
        "missing_both": int((~jpg & ~dicom).sum()),
    }
    return counts, int(len(asset_presence))


def _build_asset_mode_payload(
    joined_df: pd.DataFrame,
    *,
    category_column: str | None,
    top_n: int,
) -> dict[str, object]:
    if joined_df.empty:
        return {
            "total_count": 0,
            "overall": {"only_jpg": 0, "only_dicom": 0, "both": 0, "missing_both": 0},
            "by_value": pd.DataFrame(
                columns=["category_value", "row_count", "only_jpg", "only_dicom", "both", "missing_both"]
            ),
        }

    working = joined_df.copy()
    if category_column and category_column in working.columns:
        working["_cat_value"] = (
            working[category_column].astype(str).str.strip().replace("", "<EMPTY>")
        )
    else:
        working["_cat_value"] = "ALL"

    overall_counts, total_assets = _asset_counts(working)

    asset_category = working[["asset_id", "_cat_value"]].drop_duplicates()
    top_values = (
        asset_category["_cat_value"]
        .value_counts(dropna=False)
        .head(int(top_n))
        .index
        .tolist()
    )

    rows: list[dict[str, int | str]] = []
    for value in top_values:
        value_assets = set(
            asset_category.loc[asset_category["_cat_value"].eq(value), "asset_id"].astype(str)
        )
        subset = working.loc[working["asset_id"].astype(str).isin(value_assets)]
        counts, value_asset_count = _asset_counts(subset)
        rows.append(
            {
                "category_value": str(value),
                "row_count": int(value_asset_count),
                "only_jpg": counts["only_jpg"],
                "only_dicom": counts["only_dicom"],
                "both": counts["both"],
                "missing_both": counts["missing_both"],
            }
        )

    by_value = pd.DataFrame(
        rows,
        columns=["category_value", "row_count", "only_jpg", "only_dicom", "both", "missing_both"],
    )
    return {
        "total_count": int(total_assets),
        "overall": overall_counts,
        "by_value": by_value,
    }


_column_description_lookup = {
    region: _load_description_lookup(region)
    for region in ["Knee", "Hip"]
}

_column_display_labels = {
    region: {
        column_name: _shorten_category_label(region, column_name, _column_description_lookup.get(region, {}).get(column_name, ""))
        for column_name in semiquant_result.category_columns_by_region.get(region, [])
    }
    for region in ["Knee", "Hip"]
}


def _refresh_column_options(*_):
    region = region_toggle.value
    columns = [
        column
        for column in semiquant_result.category_columns_by_region.get(region, [])
        if "semiquant01_id" not in str(column).strip().lower()
    ]

    if columns:
        options = [
            (_column_display_labels.get(region, {}).get(column, column), column)
            for column in columns
        ]
    else:
        options = [("<no categorical columns>", _NO_CATEGORY)]

    current = column_dropdown.value
    column_dropdown.options = options
    valid_values = [value for _, value in options]
    column_dropdown.value = current if current in valid_values else options[0][1]


def _refresh_timepoint_options(*_):
    region = region_toggle.value
    joined = semiquant_result.joined_by_region.get(region)

    options = [("ALL", "ALL")]
    if joined is not None and not joined.empty:
        values = _extract_timepoint_series(joined).dropna().astype(str).unique().tolist()
        values = sorted(values, key=_timepoint_sort_key)
        options.extend([(_timepoint_option_label(value), value) for value in values])

    current = timepoint_dropdown.value
    timepoint_dropdown.options = options
    valid_values = [value for _, value in options]
    timepoint_dropdown.value = current if current in valid_values else "ALL"


def _draw_venn(*_):
    with venn_out:
        clear_output(wait=True)

        region = region_toggle.value
        joined = semiquant_result.joined_by_region.get(region)
        if joined is None or joined.empty:
            print(f"No semiquant rows available for region={region}.")
            return

        selected_timepoint = timepoint_dropdown.value
        selected_timepoint_label = _timepoint_option_label(selected_timepoint)
        if selected_timepoint == "ALL":
            joined_filtered = joined.copy()
        else:
            timepoint_series = _extract_timepoint_series(joined)
            joined_filtered = joined.loc[timepoint_series.eq(selected_timepoint)].copy()

        if joined_filtered.empty:
            print(f"No rows for region={region}, timepoint={selected_timepoint}.")
            return

        selected = column_dropdown.value
        category_column = None if selected == _NO_CATEGORY else selected

        if category_column is None:
            top_n = 1
        else:
            top_n = int(
                joined_filtered[category_column]
                .astype(str)
                .str.strip()
                .replace("", "<EMPTY>")
                .nunique(dropna=False)
            )

        count_mode = count_mode_toggle.value
        count_label = "Rows" if count_mode == "rows" else "Assets"

        if count_mode == "rows":
            payload = build_venn_payload(
                joined_filtered,
                category_column=category_column,
                top_n=max(1, top_n),
            )
            total_count = int(payload.total_rows)
            overall = {
                "only_jpg": int(payload.overall.only_jpg),
                "only_dicom": int(payload.overall.only_dicom),
                "both": int(payload.overall.both),
                "missing_both": int(payload.overall.missing_both),
            }
            by_value = payload.by_value.copy()
        else:
            asset_payload = _build_asset_mode_payload(
                joined_filtered,
                category_column=category_column,
                top_n=max(1, top_n),
            )
            total_count = int(asset_payload["total_count"])
            overall = dict(asset_payload["overall"])
            by_value = asset_payload["by_value"].copy()

        value_rows = by_value.to_dict("records")
        if category_column is not None:
            value_rows = sorted(value_rows, key=lambda row: _value_scale_key(str(row.get("category_value", ""))))

        n_sub = len(value_rows)
        ncols = 3
        nrows = 1 + max(1, math.ceil(n_sub / ncols))

        fig, axes = plt.subplots(nrows, ncols, figsize=(5.4 * ncols, 4.2 * nrows))
        axes = np.array(axes).reshape(-1)

        main_ax = axes[0]
        _draw_venn_or_message(
            main_ax,
            only_jpg=int(overall["only_jpg"]),
            only_dicom=int(overall["only_dicom"]),
            both=int(overall["both"]),
            set_colors=("#4C78A8", "#F58518"),
            alpha=0.60,
            empty_message=f"{count_label} with JPEG/DICOM: 0 | Total {count_label.lower()}: {total_count:,}",
        )
        title_column = (
            _column_display_labels.get(region, {}).get(category_column, category_column)
            if category_column
            else "ALL"
        )
        main_ax.set_title(
            f"{region} semiquant matched to downloaded images\n"
            f"Timepoint: {selected_timepoint_label} | Category: {title_column} | "
            f"Count: {'Semiquant rows' if count_mode == 'rows' else 'Unique asset_id'} | N={total_count:,}"
        )
        _annotate_missing(main_ax, int(overall["missing_both"]))

        for idx, row in enumerate(value_rows, start=1):
            ax = axes[idx]
            value_count = int(row["row_count"])
            _draw_venn_or_message(
                ax,
                only_jpg=int(row["only_jpg"]),
                only_dicom=int(row["only_dicom"]),
                both=int(row["both"]),
                set_colors=("#72B7B2", "#ECA82C"),
                alpha=0.55,
                empty_message=f"{count_label} with JPEG/DICOM: 0 | N={value_count:,}",
            )
            ax.set_title(
                f"{title_column} = {row['category_value']}\n"
                f"N={value_count:,}"
            )
            _annotate_missing(ax, int(row["missing_both"]))

        for ax in axes[(1 + n_sub):]:
            ax.axis("off")

        fig.tight_layout()
        plt.show()


_refresh_column_options()
_refresh_timepoint_options()
region_toggle.observe(_refresh_column_options, names="value")
region_toggle.observe(_refresh_timepoint_options, names="value")
region_toggle.observe(_draw_venn, names="value")
timepoint_dropdown.observe(_draw_venn, names="value")
count_mode_toggle.observe(_draw_venn, names="value")
column_dropdown.observe(_draw_venn, names="value")

display(widgets.HBox([region_toggle, timepoint_dropdown, count_mode_toggle, column_dropdown]), venn_out)
_draw_venn()



Output()

In [13]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display

semiquant_region_selector = widgets.Dropdown(
    options=["ALL", "Knee", "Hip"],
    value="ALL",
    description="Region",
)
semiquant_count_mode = widgets.ToggleButtons(
    options=[("Semiquant rows", "rows"), ("Unique asset_id", "assets")],
    value="rows",
    description="Count",
)
semiquant_table_out = widgets.Output()

_VISIT_TO_TIMEPOINT_LABEL = {
    "V00": "BASELINE",
    "V01": "12MONTH",
    "V02": "18MONTH",
    "V03": "24MONTH",
    "V05": "30MONTH",
    "V06": "48MONTH",
    "V08": "72MONTH",
    "V10": "96MONTH",
}


def _visit_sort_key(value: str) -> tuple[int, int, str]:
    text = str(value).strip().upper()
    if text.startswith("V") and text[1:].isdigit():
        return (0, int(text[1:]), text)
    if text in {"", "<EMPTY>"}:
        return (2, 0, text)
    return (1, 0, text)


def _as_bool(series: pd.Series) -> pd.Series:
    return pd.Series(series, dtype="boolean").fillna(False).astype(bool)


def _build_region_summary(region: str, *, count_mode: str) -> pd.DataFrame:
    joined = semiquant_result.joined_by_region.get(region)
    if joined is None or joined.empty:
        return pd.DataFrame(
            columns=[
                "region",
                "visit",
                "timepoint_label",
                "semiquant_total",
                "jpeg_only",
                "dicom_only",
                "both",
                "any",
                "missing_both",
                "any_pct_of_total",
            ]
        )

    frame = joined.copy()
    frame["region"] = region
    frame["visit"] = frame.get("visit", pd.Series("", index=frame.index)).astype(str).str.strip().replace("", "<EMPTY>")
    frame["asset_id"] = frame.get("asset_id", pd.Series("", index=frame.index)).astype(str).str.strip()
    frame = frame.loc[frame["asset_id"].ne("")].copy()

    frame["has_jpg"] = _as_bool(frame.get("has_jpg", False))
    frame["has_dicom"] = _as_bool(frame.get("has_dicom", False))

    if count_mode == "assets":
        frame = (
            frame.groupby(["region", "visit", "asset_id"], as_index=False)
            .agg(has_jpg=("has_jpg", "max"), has_dicom=("has_dicom", "max"))
        )

    frame["jpeg_only"] = (frame["has_jpg"] & ~frame["has_dicom"]).astype(int)
    frame["dicom_only"] = (frame["has_dicom"] & ~frame["has_jpg"]).astype(int)
    frame["both"] = (frame["has_jpg"] & frame["has_dicom"]).astype(int)
    frame["any"] = (frame["has_jpg"] | frame["has_dicom"]).astype(int)
    frame["missing_both"] = (~frame["has_jpg"] & ~frame["has_dicom"]).astype(int)

    summary = (
        frame.groupby(["region", "visit"], as_index=False)
        .agg(
            semiquant_total=("asset_id", "size"),
            jpeg_only=("jpeg_only", "sum"),
            dicom_only=("dicom_only", "sum"),
            both=("both", "sum"),
            any=("any", "sum"),
            missing_both=("missing_both", "sum"),
        )
    )
    summary["timepoint_label"] = summary["visit"].map(_VISIT_TO_TIMEPOINT_LABEL).fillna(summary["visit"])

    denom = pd.to_numeric(summary["semiquant_total"], errors="coerce")
    summary["any_pct_of_total"] = np.where(denom > 0, 100.0 * summary["any"] / denom, np.nan)
    summary["any_pct_of_total"] = summary["any_pct_of_total"].round(1)

    cols = [
        "region",
        "timepoint_label",
        "visit",
        "semiquant_total",
        "jpeg_only",
        "dicom_only",
        "both",
        "any",
        "missing_both",
        "any_pct_of_total",
    ]
    return summary[cols]


def _append_total_row(df: pd.DataFrame, *, region_label: str, total_label: str) -> pd.DataFrame:
    if df.empty:
        return df

    total = {col: pd.NA for col in df.columns}
    total["region"] = region_label
    total["timepoint_label"] = total_label
    total["visit"] = "ALL"
    if "status" in total:
        total["status"] = "summary_total"

    sum_cols = ["semiquant_total", "jpeg_only", "dicom_only", "both", "any", "missing_both"]
    for col in sum_cols:
        if col in df.columns:
            total[col] = int(pd.to_numeric(df[col], errors="coerce").fillna(0).sum())

    if "any_pct_of_total" in total:
        denom = int(total.get("semiquant_total", 0) or 0)
        total["any_pct_of_total"] = round((100.0 * int(total.get("any", 0) or 0) / denom), 1) if denom > 0 else np.nan

    return pd.concat([df, pd.DataFrame([total], columns=df.columns)], ignore_index=True)


def _collect_summary(region: str, *, count_mode: str) -> pd.DataFrame:
    knee = _build_region_summary("Knee", count_mode=count_mode)
    hip = _build_region_summary("Hip", count_mode=count_mode)
    summary = pd.concat([knee, hip], ignore_index=True)

    if region == "ALL":
        value_cols = ["semiquant_total", "jpeg_only", "dicom_only", "both", "any", "missing_both"]
        summary = (
            summary.groupby(["timepoint_label", "visit"], as_index=False)[value_cols]
            .sum()
        )
        summary["region"] = "ALL"
        denom = pd.to_numeric(summary["semiquant_total"], errors="coerce")
        summary["any_pct_of_total"] = np.where(denom > 0, 100.0 * summary["any"] / denom, np.nan)
        summary["any_pct_of_total"] = summary["any_pct_of_total"].round(1)
        summary = summary[
            [
                "region",
                "timepoint_label",
                "visit",
                "semiquant_total",
                "jpeg_only",
                "dicom_only",
                "both",
                "any",
                "missing_both",
                "any_pct_of_total",
            ]
        ]
    else:
        summary = summary.loc[summary["region"].eq(region)].copy()

    summary["_visit_order"] = summary["visit"].map(_visit_sort_key)
    summary = summary.sort_values(["_visit_order", "timepoint_label"]).drop(columns=["_visit_order"])
    summary = summary.reset_index(drop=True)
    return summary


def _found_on_disk_timepoints() -> set[str]:
    counts = inventory_result.category_counts.copy()
    if counts.empty:
        return set()

    counts["timepoint"] = counts["timepoint"].astype(str)
    counts["jpg"] = pd.to_numeric(counts["jpg"], errors="coerce").fillna(0)
    counts["dicom"] = pd.to_numeric(counts["dicom"], errors="coerce").fillna(0)

    grp = counts.groupby("timepoint", as_index=False)[["jpg", "dicom"]].sum()
    return set(grp.loc[(grp["jpg"] + grp["dicom"]) > 0, "timepoint"].astype(str))


def _fit_exclusive_rates(observed: pd.DataFrame) -> dict[str, float]:
    if observed.empty:
        return {
            "jpeg_only": 0.0,
            "dicom_only": 0.0,
            "both": 0.0,
            "missing_both": 1.0,
        }

    total = float(pd.to_numeric(observed["semiquant_total"], errors="coerce").fillna(0).sum())
    if total <= 0:
        return {
            "jpeg_only": 0.0,
            "dicom_only": 0.0,
            "both": 0.0,
            "missing_both": 1.0,
        }

    rates = {
        "jpeg_only": float(pd.to_numeric(observed["jpeg_only"], errors="coerce").fillna(0).sum()) / total,
        "dicom_only": float(pd.to_numeric(observed["dicom_only"], errors="coerce").fillna(0).sum()) / total,
        "both": float(pd.to_numeric(observed["both"], errors="coerce").fillna(0).sum()) / total,
        "missing_both": float(pd.to_numeric(observed["missing_both"], errors="coerce").fillna(0).sum()) / total,
    }

    norm = sum(rates.values())
    if norm <= 0:
        return {
            "jpeg_only": 0.0,
            "dicom_only": 0.0,
            "both": 0.0,
            "missing_both": 1.0,
        }

    return {k: float(v) / norm for k, v in rates.items()}


def _allocate_counts(total: int, rates: dict[str, float]) -> dict[str, int]:
    keys = ["jpeg_only", "dicom_only", "both", "missing_both"]
    raw = {k: max(0.0, float(rates.get(k, 0.0)) * int(total)) for k in keys}
    base = {k: int(np.floor(v)) for k, v in raw.items()}
    remainder = int(total) - int(sum(base.values()))

    if remainder > 0:
        order = sorted(keys, key=lambda k: (raw[k] - base[k]), reverse=True)
        for i in range(remainder):
            base[order[i % len(order)]] += 1
    elif remainder < 0:
        order = sorted(keys, key=lambda k: (raw[k] - base[k]))
        rem = -remainder
        idx = 0
        while rem > 0 and idx < 10000:
            k = order[idx % len(order)]
            if base[k] > 0:
                base[k] -= 1
                rem -= 1
            idx += 1

    return base


def _build_prediction_table(summary: pd.DataFrame) -> pd.DataFrame:
    pred = summary.copy()
    found_labels = _found_on_disk_timepoints()

    pred["status"] = np.where(
        pred["timepoint_label"].astype(str).isin(found_labels),
        "found_on_disk",
        "predicted_from_semiquant",
    )

    observed = pred.loc[pred["status"].eq("found_on_disk")].copy()
    rates = _fit_exclusive_rates(observed)

    for idx, row in pred.iterrows():
        total = int(pd.to_numeric(row.get("semiquant_total", 0), errors="coerce") or 0)
        if row["status"] == "found_on_disk":
            continue

        alloc = _allocate_counts(total, rates)
        pred.at[idx, "jpeg_only"] = alloc["jpeg_only"]
        pred.at[idx, "dicom_only"] = alloc["dicom_only"]
        pred.at[idx, "both"] = alloc["both"]
        pred.at[idx, "missing_both"] = alloc["missing_both"]
        pred.at[idx, "any"] = alloc["jpeg_only"] + alloc["dicom_only"] + alloc["both"]

    denom = pd.to_numeric(pred["semiquant_total"], errors="coerce")
    pred["any_pct_of_total"] = np.where(denom > 0, 100.0 * pd.to_numeric(pred["any"], errors="coerce") / denom, np.nan)
    pred["any_pct_of_total"] = pred["any_pct_of_total"].round(1)

    pred = pred[
        [
            "region",
            "timepoint_label",
            "visit",
            "status",
            "semiquant_total",
            "jpeg_only",
            "dicom_only",
            "both",
            "any",
            "missing_both",
            "any_pct_of_total",
        ]
    ]
    return pred


def _render_semiquant_table(*_):
    with semiquant_table_out:
        clear_output(wait=True)

        count_mode = semiquant_count_mode.value
        region = semiquant_region_selector.value

        full_summary = _collect_summary(region, count_mode=count_mode)

        found_labels = _found_on_disk_timepoints()
        observed = full_summary.loc[
            full_summary["timepoint_label"].astype(str).isin(found_labels)
        ].copy()
        observed_with_total = _append_total_row(
            observed,
            region_label=region,
            total_label="TOTAL_SEMIQUANT_VALID_ON_DISK",
        )

        prediction = _build_prediction_table(full_summary)
        prediction_with_total = _append_total_row(
            prediction,
            region_label=region,
            total_label="TOTAL_PREDICTED_AVAILABILITY",
        )

        print(
            "Semiquant Availability Table (On-Disk Only) "
            f"({ 'Semiquant rows' if count_mode == 'rows' else 'Unique asset_id' }; "
            f"Region={region})"
        )
        display(observed_with_total)

        print(
            "Semiquant Prediction Table "
            f"({ 'Semiquant rows' if count_mode == 'rows' else 'Unique asset_id' }; "
            f"Region={region})"
        )
        display(prediction_with_total)


semiquant_region_selector.observe(_render_semiquant_table, names="value")
semiquant_count_mode.observe(_render_semiquant_table, names="value")

display(widgets.HBox([semiquant_region_selector, semiquant_count_mode]), semiquant_table_out)
_render_semiquant_table()




Output()